# 00 — Descubrimiento de Competidores

**Ejecutar UNA vez** antes de empezar la experimentacion.

Este notebook:
1. Pregunta las 15 queries fijas a Gemini con Google Search grounding (busqueda web real)
2. Extrae las URLs que cita via grounding metadata (citations verificadas)
3. Rankea por frecuencia de aparicion
4. Permite revisar y aprobar la lista final
5. Scrapea las URLs seleccionadas
6. Genera embeddings y congela el vectorstore FAISS

**Resultado**: `data/frozen_vectorstore/` con el indice FAISS congelado

Ver ADR-004, ADR-006 y ADR-010 en `docs/DECISIONS.md`

In [ ]:
# === Bootstrap Kaggle ===
import os, sys, importlib

if os.path.exists('/kaggle'):
    # 1. Clonar / actualizar repo
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ['GOOGLE_API_KEY'] = secrets.get_secret('GOOGLE_API_KEY')
    os.environ['OPENAI_API_KEY'] = secrets.get_secret('OPENAI_API_KEY')
    os.environ['USER_AGENT'] = 'GeoAuditBot/1.0 (TFG Research)'
    token = secrets.get_secret("github_token")

    REPO_URL = f"https://{token}@github.com/angelmanuelferrer/TFG.git"
    REPO_DIR = "/kaggle/working/TFG"

    if os.path.exists(REPO_DIR):
        print("Repo ya existe, haciendo pull...")
        !cd {REPO_DIR} && git pull
    else:
        print("Clonando repo...")
        !git clone {REPO_URL} {REPO_DIR}

    # 2. Cambiar a rama kaggle
    !cd {REPO_DIR} && git checkout kaggle

    # 3. Instalar dependencias (en bloques para no colapsar CPU)
    %pip install -q "google-genai>=1,<2"
    %pip install -q "langchain>=0.3,<0.4" "langchain-community>=0.3,<0.4"
    %pip install -q "langchain-huggingface>=0.1,<1" "langchain-google-genai>=4,<5"
    %pip install -q "langchain-openai>=0.3,<0.4" "langgraph>=0.3,<0.4"
    %pip install -q "faiss-cpu>=1.9,<2" "tiktoken>=0.8,<1"
    %pip install -q "sentence-transformers>=3,<4"
    %pip install -q "beautifulsoup4>=4,<5" "requests>=2,<3" "python-dotenv>=1,<2"

    # 4. Configurar entorno — invalidar caches ANTES de insertar path
    importlib.invalidate_caches()
    sys.path.insert(0, REPO_DIR)
    os.chdir(REPO_DIR)

    print(f"Entorno Kaggle listo — {REPO_DIR}")
else:
    from dotenv import load_dotenv
    load_dotenv()
    sys.path.insert(0, os.path.abspath('..'))

# Verificacion
importlib.invalidate_caches()
print(f"sys.path[0]: {sys.path[0]}")
print(f"CWD: {os.getcwd()}")
print(f"src/ exists: {os.path.isdir(os.path.join(sys.path[0], 'src'))}")
print(f"src/__init__.py exists: {os.path.isfile(os.path.join(sys.path[0], 'src', '__init__.py'))}")
!git branch --show-current

In [ ]:
# === Importar configuracion del proyecto ===
# Celda separada para evitar problemas de cache de importacion tras pip install
import importlib
if 'src' in sys.modules:
    importlib.reload(sys.modules['src'])

from src.config import PROJECT_ROOT
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# === Cargar configuracion y queries de discovery ===
from src.config import load_experiment_config, get_discovery_queries

config = load_experiment_config()
queries = get_discovery_queries()  # Solo informacionales + comparativas

print(f"Target: {config['target_url']}")
print(f"Queries para discovery: {len(queries)} (sin navegacionales)")
for i, q in enumerate(queries, 1):
    print(f"  {i:2d}. {q}")

In [ ]:
# === Descubrir competidores desde Gemini con Google Search grounding ===
from src.discovery.competitor_finder import CompetitorFinder

finder = CompetitorFinder(config)

# delay=5.0 para respetar Gemini free tier (15 RPM).
# Discovery usa 1 llamada/query, asi que 5s = 12 RPM con margen.
results = finder.discover_competitors(queries, delay=5.0)

print(f"\nFecha: {results['discovery_date']}")
print(f"Engines: {results['engines_used']}")
print(f"Queries usadas: {results['n_queries']}")
print(f"\nTop competidores por dominio (score = citations x2 + text x1):")
for entry in results['aggregated_domains'][:20]:
    print(f"  [{entry['score']:2d}pts | {entry['n_queries']}q | {entry['citation_count']}cit] {entry['domain']}")

In [ ]:
# === Revisar y seleccionar competidores ===
# top_competitors son dominios. Seleccionar URLs representativas para scrape.
top_domains = results['top_competitors'][:10]

# Para cada dominio, incluir TODAS las URLs descubiertas (mas contenido = mejor vectorstore)
selected_urls = []
for domain in top_domains:
    domain_info = next(d for d in results['aggregated_domains'] if d['domain'] == domain)
    selected_urls.extend(domain_info['urls'])

print(f"Dominios seleccionados: {len(top_domains)}")
print(f"URLs totales a scrapear: {len(selected_urls)}")
for domain in top_domains:
    domain_info = next(d for d in results['aggregated_domains'] if d['domain'] == domain)
    print(f"\n  {domain} ({len(domain_info['urls'])} URLs):")
    for url in domain_info['urls'][:5]:
        print(f"    - {url}")
    if len(domain_info['urls']) > 5:
        print(f"    ... y {len(domain_info['urls'])-5} más")

# Guardar resultados del discovery
output_path = PROJECT_ROOT / 'config' / 'frozen_competitors.json'
finder.save_results(results, str(output_path))

In [ ]:
# === Crawl target + competidores con SiteCrawler ===
from src.processing.site_crawler import SiteCrawler
from src.processing.chunker import TokenAwareChunker

crawler = SiteCrawler(config)
chunker = TokenAwareChunker(config)
all_docs = []

# --- Target: crawl completo del sitio (sitemap → todas las paginas) ---
print(f"=== Crawling TARGET: {config['target_url']} ===")
target_raw_docs = crawler.crawl_site(config['target_url'], is_target=True)
if target_raw_docs:
    target_chunks = chunker.chunk_documents(target_raw_docs)
    all_docs.extend(target_chunks)
    print(f"  Target: {len(target_raw_docs)} pages -> {len(target_chunks)} chunks")
else:
    print("  ERROR: No se pudo crawlear el target")

# --- Competidores: discovered URLs + expansion por sitemap ---
for domain in top_domains:
    domain_info = next(d for d in results['aggregated_domains'] if d['domain'] == domain)
    discovered = domain_info['urls']
    # Usar la primera URL como punto de entrada al sitio
    entry_url = discovered[0] if discovered else f"https://{domain}"
    
    print(f"\n=== Crawling COMPETITOR: {domain} ({len(discovered)} discovered URLs) ===")
    comp_raw_docs = crawler.crawl_site(
        entry_url, is_target=False, discovered_urls=discovered
    )
    if comp_raw_docs:
        comp_chunks = chunker.chunk_documents(comp_raw_docs)
        all_docs.extend(comp_chunks)
        print(f"  {domain}: {len(comp_raw_docs)} pages -> {len(comp_chunks)} chunks")
    else:
        print(f"  {domain}: FAILED")

print(f"\nTotal chunks: {len(all_docs)}")

In [ ]:
# === Generar embeddings y crear vectorstore FAISS ===
from src.processing.embeddings import create_embeddings
from langchain_community.vectorstores import FAISS

embeddings = create_embeddings(config)

# Separar docs de competidores (se congelan) del target (se regenera cada run)
competitor_docs = [d for d in all_docs if not d.metadata.get('is_target', False)]
target_docs = [d for d in all_docs if d.metadata.get('is_target', False)]

print(f"Competitor docs: {len(competitor_docs)}")
print(f"Target docs: {len(target_docs)}")

# Crear y guardar vectorstore de competidores (ruta absoluta)
if competitor_docs:
    competitor_vs = FAISS.from_documents(competitor_docs, embeddings)
    
    vs_path = PROJECT_ROOT / 'data' / 'frozen_vectorstore'
    vs_path.mkdir(parents=True, exist_ok=True)
    competitor_vs.save_local(str(vs_path))
    print(f"\nVectorstore congelado guardado en {vs_path}/")
else:
    print("ERROR: No hay documentos de competidores para congelar")

## Siguiente paso

El vectorstore esta congelado en `data/frozen_vectorstore/`. 
Ahora ejecuta `experimental_run.ipynb` para cada run experimental.

**IMPORTANTE**: No vuelvas a ejecutar este notebook a menos que quieras
cambiar el set de competidores (lo cual invalida todos los runs anteriores).

In [ ]:
# === Commit y push a rama kaggle ===
!git config user.email "kaggle@tfg.local"
!git config user.name "TFG Kaggle Runner"
!git add data/frozen_vectorstore/ config/frozen_competitors.json
!git commit -m "data: frozen vectorstore + competitors discovery"
!git push origin kaggle